In [0]:
%sql
DROP TABLE IF EXISTS workspace.bronzeweather.years_bronzestream;

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weatherproject/checkpoint"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/commits/,commits/,0,1791121421063
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/metadata,metadata,45,1791103419000
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/offsets/,offsets/,0,1791121421063
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/sources/,sources/,0,1791121421063


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weatherproject/checkpoint/offsets/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/offsets/0,0,2302,1791103421000
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/offsets/1,1,2302,1791103422000


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weatherproject/checkpoint/sources/0/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/sources/0/metadata,metadata,133,1791103420000
dbfs:/Volumes/workspace/default/weatherproject/checkpoint/sources/0/rocksdb/,rocksdb/,0,1791122013195


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weatherproject/schema/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/weatherproject/schema/_schemas/,_schemas/,0,1791122718664


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weatherproject/schema/_schemas/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/weatherproject/schema/_schemas/0,0,705,1791100418000


In [0]:
schema_content = dbutils.fs.head("/Volumes/workspace/default/weatherproject/schema/_schemas/0")
print(schema_content)

v1
{"dataSchemaJson":"{\"type\":\"struct\",\"fields\":[{\"name\":\"_c0\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c1\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c2\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c3\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c4\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c5\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c6\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"_c7\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}}]}","partitionSchemaJson":"{\"type\":\"struct\",\"fields\":[]}"}


In [0]:

dbutils.fs.rm("/Volumes/workspace/default/weatherproject/checkpoint", True)

dbutils.fs.rm("/Volumes/workspace/default/weatherproject/schema", True)

True

In [0]:
from pyspark.sql.functions import current_timestamp
df_bronze=spark.readStream.format("cloudFiles") \
.option("cloudFiles.format", "csv") \
.option("cloudFiles.schemaEvolutionMode","rescue") \
.option("header","false")\
.option("cloudFiles.schemaLocation", "/Volumes/workspace/default/weatherproject/schema") \
.load("/Volumes/workspace/default/weatherproject/year/*.csv.gz")
df_bronze=df_bronze.select("_metadata.file_path","_metadata.file_modification_time", current_timestamp().alias ("ingesttime"),"*")
df_bronze=df_bronze.toDF("filepath","modificationtime","ingesttime","ID","DATE","ELEMENT","DATA_VALUE","M_FLAG","Q_FLAG","S_FLAG","OBS_TIME","_rescued_data")
df_bronze=df_bronze.writeStream\
.format("delta")\
.option("mergeSchema", "true")\
.outputMode("append")\
.option("checkpointLocation","/Volumes/workspace/default/weatherproject/checkpoint")\
.trigger(availableNow=True)\
.toTable("workspace.bronzeweather.years_bronzestream")



In [0]:
%sql
with t as (select count(*) as cnt  from workspace.bronzeweather.years_bronzestream group by ID, DATE, ELEMENT) select * from t where cnt  > 1

cnt


In [0]:
%sql
describe history workspace.bronzeweather.years_bronzestream

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-04T14:30:27.000Z,76285942558036,melikakhajeh94@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 48048958-49b9-4e90-9b45-c82fecaeb3ff, epochId -> 1, statsOnLoad -> true)",null,List(1782910737545685),c86da337-89ce-48f2-94cc-39057cd4ad53,1004-132959-rfvukbpk-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 71, numOutputRows -> 35058603, numOutputBytes -> 70632510, numAddedFiles -> 1)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-04T14:30:25.000Z,76285942558036,melikakhajeh94@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 48048958-49b9-4e90-9b45-c82fecaeb3ff, epochId -> 0, statsOnLoad -> true)",null,List(1782910737545685),c86da337-89ce-48f2-94cc-39057cd4ad53,1004-132959-rfvukbpk-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 37137956, numOutputBytes -> 78182008, numAddedFiles -> 1)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-04T14:28:43.000Z,76285942558036,melikakhajeh94@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""de45ca50-1caf-438e-98d9-96e0d16fd9cc"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-001c3b79-423e-40f4-a95d-21415969bab0"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-50b6f3f2-334e-4aeb-ae68-ec32505b63ae""}, statsOnLoad -> false)",null,List(1782910737545685),c86da337-89ce-48f2-94cc-39057cd4ad53,1004-132959-rfvukbpk-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
%sql
select count(*) from workspace.bronzeweather.years_bronzestream


count(*)
72196559
